In [1]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torch.nn import *
from torch.optim import *
from torchvision.models import *
from sklearn.model_selection import *
from sklearn.metrics import *
import wandb
import nltk
from nltk.stem.porter import *
PROJECT_NAME = "Natural-Language-Processing-with-Disaster-Tweets"
np.random.seed(55)
stemmer = PorterStemmer()

In [2]:
def tokenize(sentence):
    return nltk.word_tokenize(sentence.lower())

In [3]:
tokenize("Testing this tokenize function")

['testing', 'this', 'tokenize', 'function']

In [4]:
def stem(word):
    return stemmer.stem(word.lower())

In [5]:
stem('organic')

'organ'

In [6]:
def words_to_int(words,all_words):
    new_words = []
    for word in words:
        new_words.append(stem(word))
    list_of_os = np.zeros(len(all_words))
    for i in range(len(all_words)):
        if all_words[i] in new_words:
            list_of_os[i] = 1.0
    return list_of_os

In [7]:
words_to_int(["test"],["testing","I","test","grswgre"])

array([0., 0., 1., 0.])

In [8]:
test = pd.read_csv('./data/test.csv')

In [9]:
data = pd.read_csv('./data/train.csv')
data = data.sample(frac=1)
data = data.sample(frac=1)
data = data.sample(frac=1)

In [10]:
data

,id,keyword,location,text,target
5801,8280,rioting,"heart of darkness, unholy ?",@Georgous__ what alternatives? Legal alternati...,0
5112,7291,nuclear%20disaster,NaN,#Nuclear policy of #Japan without responsibili...,1
5082,7248,nuclear%20disaster,Austin TX,Alarming Rise in Dead Marine Life Since the #F...,1
804,1167,blight,"Vancouver, BC",@parksboardfacts first off it is the #ZippoLin...,0
6451,9230,suicide%20bombing,Memphis,Kurd Suicide Attack Kills 2 Turkish Soldiers h...,1
...,...,...,...,...,...
7267,10405,whirlwind,"Harbour Heights, FL",@DrMartyFox In the U.S. government and Lib...,0
1596,2304,collapse,NaN,Runaway Minion Causes Traffic Collapse in Dubl...,1
5,8,NaN,NaN,#RockyFire Update => California Hwy. 20 closed...,1
4619,6567,injury,NaN,FollowNFLNews: Michael Floyd's hand injury sho...,0


In [11]:
# TODO Use Keyboard / Location in X

In [12]:
X = data['text']
y = data['target']

In [13]:
all_words = []
tags = []

In [14]:
from tqdm import tqdm

In [15]:
for x_iter,y_iter in tqdm(zip(X,y)):
    x_iter = tokenize(x_iter)
    new_x_iter = []
    for x_iter_i in x_iter:
        new_x_iter.append(stem(x_iter_i))
    all_words.extend(new_x_iter)
    tags.append(y_iter)

7613it [00:02, 3536.36it/s]


In [16]:
np.random.shuffle(all_words)
np.random.shuffle(all_words)

In [17]:
all_words = sorted(set(all_words))

In [18]:
tags = sorted(set(tags))

In [19]:
new_X = []
new_y = []

In [20]:
for X_iter,y_iter in tqdm(zip(X,y)):
    new_X.append(words_to_int(X_iter,all_words))
    new_y.append(tags.index(y_iter))

7613it [01:33, 81.28it/s]


In [21]:
new_X[0]

array([0., 0., 0., ..., 0., 0., 0.])

In [22]:
new_y[85]

1

In [23]:
X = np.array(new_X)
y = np.array(new_y)

In [24]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.0625,shuffle=True)

In [25]:
device = 'cuda'

In [26]:
X_train = torch.from_numpy(X_train).to(device)
y_train = torch.from_numpy(y_train).to(device)
X_test = torch.from_numpy(X_test).to(device)
y_test = torch.from_numpy(y_test).to(device)

In [29]:
# class Model(Module):
#     def __init__(self):
#         super().__init__()
#         self.activation = ReLU()
#         self.output_activation = Sigmoid()
#         self.linear1 = Linear(len(all_words),512)
#         self.linear2 = Linear(512,1024)
#         self.linear3bn = BatchNorm1d(1024)
#         self.linear4 = Linear(1024,2048)
#         self.linear5 = Linear(2048,1024)
#         self.output = Linear(1024,1)
    
#     def forward(self,X):
#         preds = self.activation(self.linear1(X))
#         preds = self.activation(self.linear2(preds))
#         preds = self.activation(self.linear3bn(preds))
#         preds = self.activation(self.linear4(preds))
#         preds = self.activation(self.linear5(preds))
#         preds = self.output_activation(self.output(preds))
#         return preds
class Model(Module):
    def __init__(self):
        super().__init__()
        self.activation = ReLU()
        self.iters = 2
        self.dropout = Dropout()
        self.hidden = 512
        self.linear1 = Linear(len(all_words),self.hidden)
        self.linear2 = Linear(self.hidden,self.hidden)
        self.linear3 = Linear(self.hidden,self.hidden)
        self.linear4 = Linear(self.hidden,self.hidden)
        self.linear5 = Linear(self.hidden,self.hidden)
        self.output = Linear(self.hidden,1)
    
    def forward(self,X):
        preds = self.linear1(X)
        preds = self.activation(self.linear2(preds))
        for _ in range(self.iters):
            preds = self.dropout(self.activation(self.linear3(preds)))
        preds = self.activation(self.linear4(preds))
        preds = self.activation(self.linear5(preds))
        preds = self.output(preds)
        return preds

In [30]:
model = Model().to(device)
criterion = MSELoss()
optimizer = Adam(model.parameters(),lr=0.001)
epochs = 100
batch_size = 32

In [31]:
def accuracy(model,X,y):
    correct = 0
    total = 0
    preds = model(X.float())
    for pred,y_batch in zip(preds,y):
        pred = int(torch.argmax(pred))
        y_batch = int(torch.argmax(y_batch))
        if pred == y_batch:
            correct += 1
        total += 1
    acc = round(correct/total,3)*100
    return acc

In [32]:
def g_loss(model,X,y):
    preds = model(X.float())
    loss = criterion(preds.view(-1),y.float().view(-1))
    return loss.item()

In [ ]:
wandb.init(project=PROJECT_NAME,name='BaseLine')
wandb.watch(model)
for _ in tqdm(range(epochs)):
    for i in range(0,len(X_train),batch_size):
        try:
            X_batch = X_train[i:i+batch_size]
            y_batch = y_train[i:i+batch_size]
            preds = model(X_batch.float())
            loss = criterion(preds.float().view(-1),y_batch.float().view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        except:
            pass
    wandb.log(
        {
            "Val Accuracy":accuracy(model,X_test,y_test),
            "Val Loss":g_loss(model,X_test,y_test),
            "Accuracy":accuracy(model,X_train,y_train),
            "Loss":g_loss(model,X_train,y_train)
        }
    )
    wandb.watch(model)
wandb.watch(model)
wandb.finish()

wandb: Currently logged in as: ranuga-d (use `wandb login --relogin` to force relogin)
RuntimeError: module compiled against API version 0xe but this version of numpy is 0xd


 84%|████████████████████████████▌     | 84/100 [01:58<00:25,  1.60s/it]

In [ ]:
torch.save(X_train,'X_train.pt')
torch.save(X_test,'X_test.pth')
torch.save(y_train,'y_train.pt')
torch.save(y_test,'y_test.pth')
torch.save(model,'model.pt')
torch.save(model,'model.pth')
torch.save(model.state_dict(),'model-sd.pt')
torch.save(model.state_dict(),'model-sd.pth')
torch.save(X,'X.pt')
torch.save(X,'X.pth')
torch.save(y,'y.pt')
torch.save(y,'y.pth')